In [1]:

import sys
print("Python executable:", sys.executable)
print("Python version:", sys.version)

Python executable: C:\python\python.exe
Python version: 3.14.6 (tags/v3.14.6:c63aec6, Jun 10 2026, 10:26:10) [MSC v.1944 64 bit (AMD64)]


In [2]:

from pathlib import Path

PROJECT_ROOT = Path(r"C:\aahana\NTA")

folders = [
    "backend",
    "backend/api",
    "backend/capture",
    "backend/analyzer",
    "backend/database",
    "data",
    "logs",
    "frontend",
]

for folder in folders:
    (PROJECT_ROOT / folder).mkdir(parents=True, exist_ok=True)

print("Project folders created successfully.")
print("Project root:", PROJECT_ROOT)

Project folders created successfully.
Project root: C:\aahana\NTA


In [3]:

from scapy.all import conf, get_if_list, get_if_addr

print("Available network interfaces:\n")

for iface in conf.ifaces.values():
    try:
        print(
            "Name:", iface.name,
            "| Description:", iface.description,
            "| IP:", iface.ip
        )
    except Exception as e:
        print("Interface:", iface.name, "| Error:", e)


Available network interfaces:

Name: Local Area Connection* 8 | Description: WAN Miniport (Network Monitor) | IP: 
Name: Local Area Connection* 7 | Description: WAN Miniport (IPv6) | IP: 
Name: Local Area Connection* 6 | Description: WAN Miniport (IP) | IP: 
Name: vEthernet (WSL (Hyper-V firewall)) | Description: Hyper-V Virtual Ethernet Adapter | IP: 172.28.112.1
Name: Bluetooth Network Connection | Description: Bluetooth Device (Personal Area Network) | IP: 169.254.158.199
Name: Wi-Fi | Description: Realtek 8852BE-VT Wireless LAN WiFi 6 PCI-E NIC | IP: 192.168.27.106
Name: Wi-Fi 4 | Description: Realtek 8852BE-VT Wireless LAN WiFi 6 PCI-E NIC #4 | IP: 169.254.133.0
Name: Wi-Fi 3 | Description: Realtek 8852BE-VT Wireless LAN WiFi 6 PCI-E NIC #3 | IP: 169.254.224.178
Name: Loopback Pseudo-Interface 1 | Description: Software Loopback Interface 1 | IP: 127.0.0.1
Name: Ethernet | Description: Realtek Gaming GbE Family Controller | IP: 172.17.5.100
Name: McAfee VPN | Description: TAP-Windo

In [4]:

from scapy.all import sniff, IP, TCP, UDP
from collections import Counter

print("Capturing packets on Wi-Fi for 10 seconds...")
print("Browse a website during this test to generate traffic.\n")

packets = sniff(
    iface="Wi-Fi",
    timeout=10,
    store=True
)

print("Capture completed.")
print("Total packets captured:", len(packets))

protocol_counts = Counter()

for packet in packets:
    if IP in packet:
        if TCP in packet:
            protocol_counts["TCP"] += 1
        elif UDP in packet:
            protocol_counts["UDP"] += 1
        else:
            protocol_counts["Other IP"] += 1
    else:
        protocol_counts["Non-IP"] += 1

print("\nProtocol counts:")
for protocol, count in protocol_counts.items():
    print(f"{protocol}: {count}")


Capturing packets on Wi-Fi for 10 seconds...
Browse a website during this test to generate traffic.

Capture completed.
Total packets captured: 132

Protocol counts:
TCP: 120
UDP: 12


In [5]:

from datetime import datetime
from scapy.all import IP, TCP, UDP, ICMP, DNS

def decode_packet(packet):
    """Extract useful network information from one packet."""

    info = {
        "timestamp": datetime.fromtimestamp(
            float(packet.time)
        ).isoformat(timespec="milliseconds"),
        "src_ip": None,
        "dst_ip": None,
        "protocol": "Other",
        "src_port": None,
        "dst_port": None,
        "packet_size": len(packet),
    }

    if IP in packet:
        info["src_ip"] = packet[IP].src
        info["dst_ip"] = packet[IP].dst

        if TCP in packet:
            info["protocol"] = "TCP"
            info["src_port"] = packet[TCP].sport
            info["dst_port"] = packet[TCP].dport
        elif UDP in packet:
            info["protocol"] = "UDP"
            info["src_port"] = packet[UDP].sport
            info["dst_port"] = packet[UDP].dport
        elif ICMP in packet:
            info["protocol"] = "ICMP"
        else:
            info["protocol"] = "Other IP"

    elif DNS in packet:
        info["protocol"] = "DNS"

    return info


# Decode the packets captured in Step 9
decoded_packets = [decode_packet(packet) for packet in packets]

print("Decoded packets:", len(decoded_packets))
print("\nFirst 5 decoded packets:")

for packet_info in decoded_packets[:5]:
    print(packet_info)


Decoded packets: 132

First 5 decoded packets:
{'timestamp': '2026-10-15T23:57:56.983', 'src_ip': '172.64.148.235', 'dst_ip': '192.168.27.106', 'protocol': 'TCP', 'src_port': 443, 'dst_port': 51831, 'packet_size': 78}
{'timestamp': '2026-10-15T23:57:56.983', 'src_ip': '192.168.27.106', 'dst_ip': '172.64.148.235', 'protocol': 'TCP', 'src_port': 51831, 'dst_port': 443, 'packet_size': 82}
{'timestamp': '2026-10-15T23:57:56.988', 'src_ip': '172.64.148.235', 'dst_ip': '192.168.27.106', 'protocol': 'TCP', 'src_port': 443, 'dst_port': 51831, 'packet_size': 60}
{'timestamp': '2026-10-15T23:57:58.932', 'src_ip': '192.168.27.106', 'dst_ip': '192.168.5.245', 'protocol': 'UDP', 'src_port': 61820, 'dst_port': 53, 'packet_size': 96}
{'timestamp': '2026-10-15T23:57:58.932', 'src_ip': '192.168.27.106', 'dst_ip': '192.168.5.245', 'protocol': 'UDP', 'src_port': 62403, 'dst_port': 53, 'packet_size': 96}


In [6]:

import csv
import json
import sqlite3
import pandas as pd
from pathlib import Path
from collections import Counter, defaultdict
from datetime import datetime

print("Required libraries imported successfully.")


Required libraries imported successfully.


In [7]:

from pathlib import Path

PROJECT_ROOT = Path(r"C:\aahana\NTA")
DATA_DIR = PROJECT_ROOT / "data"
LOG_DIR = PROJECT_ROOT / "logs"
BACKEND_DIR = PROJECT_ROOT / "backend"

for folder in [DATA_DIR, LOG_DIR, BACKEND_DIR]:
    folder.mkdir(parents=True, exist_ok=True)

CSV_PATH = DATA_DIR / "packets.csv"
DB_PATH = DATA_DIR / "network_traffic.db"

print("Project root:", PROJECT_ROOT)
print("CSV path:", CSV_PATH)
print("Database path:", DB_PATH)


Project root: C:\aahana\NTA
CSV path: C:\aahana\NTA\data\packets.csv
Database path: C:\aahana\NTA\data\network_traffic.db


In [8]:

packets_df = pd.DataFrame(decoded_packets)

print("DataFrame shape:", packets_df.shape)
print("\nColumns:")
print(packets_df.columns.tolist())

display(packets_df.head(10))


DataFrame shape: (132, 7)

Columns:
['timestamp', 'src_ip', 'dst_ip', 'protocol', 'src_port', 'dst_port', 'packet_size']


,timestamp,src_ip,dst_ip,protocol,src_port,dst_port,packet_size
0,2026-10-15T23:57:56.983,172.64.148.235,192.168.27.106,TCP,443,51831,78
1,2026-10-15T23:57:56.983,192.168.27.106,172.64.148.235,TCP,51831,443,82
2,2026-10-15T23:57:56.988,172.64.148.235,192.168.27.106,TCP,443,51831,60
3,2026-10-15T23:57:58.932,192.168.27.106,192.168.5.245,UDP,61820,53,96
4,2026-10-15T23:57:58.932,192.168.27.106,192.168.5.245,UDP,62403,53,96
5,2026-10-15T23:57:58.941,192.168.5.245,192.168.27.106,UDP,53,61820,207
6,2026-10-15T23:57:58.941,192.168.5.245,192.168.27.106,UDP,53,62403,223
7,2026-10-15T23:57:58.942,192.168.27.106,20.42.65.90,TCP,54499,443,66
8,2026-10-15T23:57:58.944,20.42.65.90,192.168.27.106,TCP,443,54499,66
9,2026-10-15T23:57:58.944,192.168.27.106,20.42.65.90,TCP,54499,443,54


In [9]:

packets_df.to_csv(CSV_PATH, index=False, encoding="utf-8")

print("CSV export completed.")
print("File exists:", CSV_PATH.exists())
print("File size:", f"{CSV_PATH.stat().st_size:,} bytes")


CSV export completed.
File exists: True
File size: 9,236 bytes


In [10]:

with sqlite3.connect(DB_PATH) as conn:
    conn.execute("""
        CREATE TABLE IF NOT EXISTS packets (
            id INTEGER PRIMARY KEY AUTOINCREMENT,
            timestamp TEXT NOT NULL,
            src_ip TEXT,
            dst_ip TEXT,
            protocol TEXT,
            src_port INTEGER,
            dst_port INTEGER,
            packet_size INTEGER
        )
    """)

    conn.execute("""
        CREATE INDEX IF NOT EXISTS idx_packets_timestamp
        ON packets(timestamp)
    """)

    conn.execute("""
        CREATE INDEX IF NOT EXISTS idx_packets_protocol
        ON packets(protocol)
    """)

print("SQLite database and packets table are ready.")


SQLite database and packets table are ready.


In [11]:

columns = [
    "timestamp", "src_ip", "dst_ip", "protocol",
    "src_port", "dst_port", "packet_size"
]

with sqlite3.connect(DB_PATH) as conn:
    packets_df[columns].to_sql(
        "packets",
        conn,
        if_exists="append",
        index=False
    )

    total_rows = conn.execute(
        "SELECT COUNT(*) FROM packets"
    ).fetchone()[0]

print("Current batch inserted:", len(packets_df))
print("Total packets stored in database:", total_rows)


Current batch inserted: 132
Total packets stored in database: 132


In [12]:

protocol_stats = (
    packets_df.groupby("protocol")
    .agg(
        packet_count=("protocol", "size"),
        total_bytes=("packet_size", "sum"),
        average_packet_size=("packet_size", "mean")
    )
    .reset_index()
    .sort_values("packet_count", ascending=False)
)

protocol_stats["average_packet_size"] = (
    protocol_stats["average_packet_size"].round(2)
)

display(protocol_stats)

print("\nProtocol statistics calculated successfully.")


,protocol,packet_count,total_bytes,average_packet_size
0,TCP,120,33419,278.49
1,UDP,12,1384,115.33



Protocol statistics calculated successfully.


In [13]:

top_sources = (
    packets_df.groupby("src_ip")
    .agg(
        packets_sent=("src_ip", "size"),
        bytes_sent=("packet_size", "sum")
    )
    .sort_values("packets_sent", ascending=False)
    .head(10)
    .reset_index()
)

top_destinations = (
    packets_df.groupby("dst_ip")
    .agg(
        packets_received=("dst_ip", "size"),
        bytes_received=("packet_size", "sum")
    )
    .sort_values("packets_received", ascending=False)
    .head(10)
    .reset_index()
)

print("Top source IPs:")
display(top_sources)

print("Top destination IPs:")
display(top_destinations)


Top source IPs:


,src_ip,packets_sent,bytes_sent
0,192.168.27.106,64,17216
1,20.42.65.90,21,10685
2,35.190.80.1,16,3189
3,104.18.32.47,6,1362
4,8.8.8.8,6,372
5,192.168.5.245,6,874
6,184.28.173.35,5,547
7,104.18.39.21,4,276
8,172.64.148.235,2,138
9,172.64.155.209,2,144


Top destination IPs:


,dst_ip,packets_received,bytes_received
0,192.168.27.106,68,17587
1,20.42.65.90,17,4979
2,35.190.80.1,15,4412
3,8.8.8.8,8,4262
4,184.28.173.35,6,447
5,104.18.32.47,6,2169
6,192.168.5.245,6,510
7,172.64.155.209,3,191
8,104.18.39.21,2,164
9,172.64.148.235,1,82


In [14]:

flow_data = packets_df.copy()

flow_data["endpoint_a"] = list(zip(
    flow_data["src_ip"].fillna(""),
    flow_data["src_port"].fillna(0).astype(int)
))
flow_data["endpoint_b"] = list(zip(
    flow_data["dst_ip"].fillna(""),
    flow_data["dst_port"].fillna(0).astype(int)
))

def make_flow_key(row):
    endpoints = sorted([row["endpoint_a"], row["endpoint_b"]])
    return (row["protocol"], endpoints[0], endpoints[1])

flow_data["flow_key"] = flow_data.apply(make_flow_key, axis=1)

flows_df = (
    flow_data.groupby("flow_key")
    .agg(
        packet_count=("packet_size", "size"),
        total_bytes=("packet_size", "sum"),
        first_seen=("timestamp", "min"),
        last_seen=("timestamp", "max")
    )
    .reset_index()
)

flows_df["protocol"] = flows_df["flow_key"].apply(lambda x: x[0])
flows_df["endpoint_a"] = flows_df["flow_key"].apply(lambda x: str(x[1]))
flows_df["endpoint_b"] = flows_df["flow_key"].apply(lambda x: str(x[2]))
flows_df.drop(columns=["flow_key"], inplace=True)

print("Unique bidirectional flows:", len(flows_df))
display(flows_df.head(10))


Unique bidirectional flows: 17


,packet_count,total_bytes,first_seen,last_seen,protocol,endpoint_a,endpoint_b
0,12,3531,2026-10-15T23:58:01.172,2026-10-15T23:58:02.113,TCP,"('104.18.32.47', 443)","('192.168.27.106', 60682)"
1,3,220,2026-10-15T23:58:03.474,2026-10-15T23:58:03.477,TCP,"('104.18.39.21', 443)","('192.168.27.106', 58170)"
2,3,220,2026-10-15T23:58:06.722,2026-10-15T23:58:06.726,TCP,"('104.18.39.21', 443)","('192.168.27.106', 58370)"
3,3,220,2026-10-15T23:57:56.983,2026-10-15T23:57:56.988,TCP,"('172.64.148.235', 443)","('192.168.27.106', 51831)"
4,2,121,2026-10-15T23:58:03.191,2026-10-15T23:58:03.205,TCP,"('172.64.155.209', 443)","('192.168.27.106', 50268)"
5,3,214,2026-10-15T23:58:04.414,2026-10-15T23:58:04.510,TCP,"('172.64.155.209', 443)","('192.168.27.106', 60101)"
6,11,994,2026-10-15T23:57:59.262,2026-10-15T23:57:59.423,TCP,"('184.28.173.35', 80)","('192.168.27.106', 54500)"
7,38,15664,2026-10-15T23:57:58.942,2026-10-15T23:58:00.779,TCP,"('192.168.27.106', 54499)","('20.42.65.90', 443)"
8,31,7601,2026-10-15T23:58:02.070,2026-10-15T23:58:02.707,TCP,"('192.168.27.106', 56497)","('35.190.80.1', 443)"
9,7,2396,2026-10-15T23:58:05.252,2026-10-15T23:58:05.273,TCP,"('192.168.27.106', 58872)","('8.8.8.8', 443)"


In [15]:

FLOWS_CSV_PATH = DATA_DIR / "flows.csv"

flows_df.to_csv(FLOWS_CSV_PATH, index=False, encoding="utf-8")

# Add a separate table for flow summaries.
with sqlite3.connect(DB_PATH) as conn:
    conn.execute("""
        CREATE TABLE IF NOT EXISTS flows (
            id INTEGER PRIMARY KEY AUTOINCREMENT,
            protocol TEXT,
            endpoint_a TEXT,
            endpoint_b TEXT,
            packet_count INTEGER,
            total_bytes INTEGER,
            first_seen TEXT,
            last_seen TEXT
        )
    """)

    flows_df.to_sql(
        "flows",
        conn,
        if_exists="replace",
        index=False
    )

    packet_count = conn.execute(
        "SELECT COUNT(*) FROM packets"
    ).fetchone()[0]

    flow_count = conn.execute(
        "SELECT COUNT(*) FROM flows"
    ).fetchone()[0]

print("Flow CSV saved:", FLOWS_CSV_PATH.exists())
print("Database exists:", DB_PATH.exists())
print("Packets in database:", packet_count)
print("Flow summaries in database:", flow_count)
print("\nData-storage phase completed.")


Flow CSV saved: True
Database exists: True
Packets in database: 132
Flow summaries in database: 17

Data-storage phase completed.


In [16]:

import fastapi
import uvicorn
import scapy

print("FastAPI:", fastapi.__version__)
print("Uvicorn:", uvicorn.__version__)
print("Scapy:", scapy.__version__)
print("Backend dependencies are available.")


FastAPI: 0.143.2
Uvicorn: 0.51.0
Scapy: 2.8.0
Backend dependencies are available.


In [17]:

backend_packages = [
    "backend/__init__.py",
    "backend/api/__init__.py",
    "backend/capture/__init__.py",
    "backend/analyzer/__init__.py",
    "backend/database/__init__.py",
]

for relative_path in backend_packages:
    path = PROJECT_ROOT / relative_path
    path.parent.mkdir(parents=True, exist_ok=True)
    path.touch(exist_ok=True)

print("Backend package files created.")


Backend package files created.


In [18]:

database_code = '''
import sqlite3
from pathlib import Path

PROJECT_ROOT = Path(__file__).resolve().parents[2]
DB_PATH = PROJECT_ROOT / "data" / "network_traffic.db"

def get_connection():
    DB_PATH.parent.mkdir(parents=True, exist_ok=True)
    conn = sqlite3.connect(DB_PATH, timeout=10)
    conn.row_factory = sqlite3.Row
    return conn

def initialize_database():
    with get_connection() as conn:
        conn.execute("""
            CREATE TABLE IF NOT EXISTS packets (
                id INTEGER PRIMARY KEY AUTOINCREMENT,
                timestamp TEXT NOT NULL,
                src_ip TEXT,
                dst_ip TEXT,
                protocol TEXT,
                src_port INTEGER,
                dst_port INTEGER,
                packet_size INTEGER
            )
        """)
        conn.execute("""
            CREATE INDEX IF NOT EXISTS idx_packets_timestamp
            ON packets(timestamp)
        """)
        conn.execute("""
            CREATE INDEX IF NOT EXISTS idx_packets_protocol
            ON packets(protocol)
        """)

def get_packet_count():
    with get_connection() as conn:
        return conn.execute(
            "SELECT COUNT(*) FROM packets"
        ).fetchone()[0]
'''

database_file = BACKEND_DIR / "database" / "db.py"
database_file.write_text(database_code.strip() + "\n", encoding="utf-8")

print("Database module created:", database_file)


Database module created: C:\aahana\NTA\backend\database\db.py


In [19]:

packet_service_code = '''
from backend.database.db import get_connection

def get_recent_packets(limit=100, protocol=None):
    limit = max(1, min(int(limit), 1000))

    query = """
        SELECT id, timestamp, src_ip, dst_ip, protocol,
               src_port, dst_port, packet_size
        FROM packets
    """
    params = []

    if protocol:
        query += " WHERE protocol = ?"
        params.append(protocol.upper())

    query += " ORDER BY id DESC LIMIT ?"
    params.append(limit)

    with get_connection() as conn:
        rows = conn.execute(query, params).fetchall()
        return [dict(row) for row in rows]

def get_protocol_summary():
    with get_connection() as conn:
        rows = conn.execute("""
            SELECT protocol,
                   COUNT(*) AS packet_count,
                   SUM(packet_size) AS total_bytes
            FROM packets
            GROUP BY protocol
            ORDER BY packet_count DESC
        """).fetchall()

    return [dict(row) for row in rows]
'''

packet_service_file = BACKEND_DIR / "analyzer" / "packet_service.py"
packet_service_file.write_text(
    packet_service_code.strip() + "\n",
    encoding="utf-8"
)

print("Packet query service created:", packet_service_file)


Packet query service created: C:\aahana\NTA\backend\analyzer\packet_service.py


In [20]:

api_code = '''
from contextlib import asynccontextmanager
from fastapi import FastAPI, Query
from fastapi.middleware.cors import CORSMiddleware

from backend.database.db import initialize_database, get_packet_count
from backend.analyzer.packet_service import (
    get_recent_packets,
    get_protocol_summary,
)

@asynccontextmanager
async def lifespan(app: FastAPI):
    initialize_database()
    yield

app = FastAPI(
    title="Network Traffic Analyzer API",
    version="1.0.0",
    lifespan=lifespan,
)

app.add_middleware(
    CORSMiddleware,
    allow_origins=[
        "http://localhost:5173",
        "http://127.0.0.1:5173",
    ],
    allow_credentials=True,
    allow_methods=["*"],
    allow_headers=["*"],
)

@app.get("/")
def root():
    return {
        "message": "Network Traffic Analyzer API",
        "docs": "/docs",
    }

@app.get("/api/health")
def health():
    return {"status": "healthy"}

@app.get("/api/stats")
def stats():
    return {
        "total_packets": get_packet_count(),
        "protocols": get_protocol_summary(),
    }

@app.get("/api/packets")
def packets(
    limit: int = Query(default=100, ge=1, le=1000),
    protocol: str | None = None,
):
    return get_recent_packets(limit=limit, protocol=protocol)
'''

api_file = BACKEND_DIR / "main.py"
api_file.write_text(api_code.strip() + "\n", encoding="utf-8")

print("FastAPI application created:", api_file)


FastAPI application created: C:\aahana\NTA\backend\main.py


In [21]:

launcher_code = '''
import uvicorn

if __name__ == "__main__":
    uvicorn.run(
        "backend.main:app",
        host="127.0.0.1",
        port=8000,
        reload=False,
    )
'''

launcher_file = BACKEND_DIR / "run.py"
launcher_file.write_text(launcher_code.strip() + "\n", encoding="utf-8")

print("Backend launcher created:", launcher_file)


Backend launcher created: C:\aahana\NTA\backend\run.py


In [22]:

import sys

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from backend.database.db import initialize_database, get_packet_count

initialize_database()

print("Database initialized through backend module.")
print("Existing packet records:", get_packet_count())


Database initialized through backend module.
Existing packet records: 132


In [23]:

from backend.analyzer.packet_service import (
    get_recent_packets,
    get_protocol_summary,
)

recent_packets = get_recent_packets(limit=5)
protocol_summary = get_protocol_summary()

print("Recent packet records:", len(recent_packets))
for packet in recent_packets:
    print(packet)

print("\nProtocol summary:")
for item in protocol_summary:
    print(item)


Recent packet records: 5
{'id': 132, 'timestamp': '2026-10-15T23:58:06.726', 'src_ip': '104.18.39.21', 'dst_ip': '192.168.27.106', 'protocol': 'TCP', 'src_port': 443, 'dst_port': 58370, 'packet_size': 60}
{'id': 131, 'timestamp': '2026-10-15T23:58:06.723', 'src_ip': '192.168.27.106', 'dst_ip': '104.18.39.21', 'protocol': 'TCP', 'src_port': 58370, 'dst_port': 443, 'packet_size': 82}
{'id': 130, 'timestamp': '2026-10-15T23:58:06.722', 'src_ip': '104.18.39.21', 'dst_ip': '192.168.27.106', 'protocol': 'TCP', 'src_port': 443, 'dst_port': 58370, 'packet_size': 78}
{'id': 129, 'timestamp': '2026-10-15T23:58:05.273', 'src_ip': '8.8.8.8', 'dst_ip': '192.168.27.106', 'protocol': 'TCP', 'src_port': 443, 'dst_port': 58872, 'packet_size': 60}
{'id': 128, 'timestamp': '2026-10-15T23:58:05.273', 'src_ip': '8.8.8.8', 'dst_ip': '192.168.27.106', 'protocol': 'TCP', 'src_port': 443, 'dst_port': 58872, 'packet_size': 60}

Protocol summary:
{'protocol': 'TCP', 'packet_count': 120, 'total_bytes': 33419}
{'p

In [24]:

from fastapi.testclient import TestClient
from backend.main import app

with TestClient(app) as client:
    root_response = client.get("/")
    health_response = client.get("/api/health")
    stats_response = client.get("/api/stats")
    packets_response = client.get("/api/packets?limit=5")

    print("Root:", root_response.status_code, root_response.json())
    print("Health:", health_response.status_code, health_response.json())
    print("Stats status:", stats_response.status_code)
    print("Stats data:", stats_response.json())
    print("Packets status:", packets_response.status_code)
    print("Packets returned:", len(packets_response.json()))


C:\python\Lib\site-packages\fastapi\testclient.py:1: StarletteDeprecationWarning: Using `httpx` with `starlette.testclient` is deprecated; install `httpx2` instead.
  from starlette.testclient import TestClient as TestClient  # noqa


Root: 200 {'message': 'Network Traffic Analyzer API', 'docs': '/docs'}
Health: 200 {'status': 'healthy'}
Stats status: 200
Stats data: {'total_packets': 132, 'protocols': [{'protocol': 'TCP', 'packet_count': 120, 'total_bytes': 33419}, {'protocol': 'UDP', 'packet_count': 12, 'total_bytes': 1384}]}
Packets status: 200
Packets returned: 5


In [25]:
%pip install httpx

[notice] A new release of pip is available: 26.1.2 -> 26.2.1


[notice] To update, run: python.exe -m pip install --upgrade pip


Note: you may need to restart the kernel to use updated packages.


In [26]:

backend_files = [
    PROJECT_ROOT / "backend" / "__init__.py",
    PROJECT_ROOT / "backend" / "main.py",
    PROJECT_ROOT / "backend" / "run.py",
    PROJECT_ROOT / "backend" / "database" / "db.py",
    PROJECT_ROOT / "backend" / "analyzer" / "packet_service.py",
]

for path in backend_files:
    print(
        "[OK]" if path.exists() else "[MISSING]",
        path.relative_to(PROJECT_ROOT)
    )

print("\nBackend API files verified.")

[OK] backend\__init__.py
[OK] backend\main.py
[OK] backend\run.py
[OK] backend\database\db.py
[OK] backend\analyzer\packet_service.py

Backend API files verified.


In [27]:

decoder_code = '''
from datetime import datetime
from scapy.all import IP, TCP, UDP, ICMP, IPv6

def decode_packet(packet):
    info = {
        "timestamp": datetime.fromtimestamp(
            float(packet.time)
        ).isoformat(timespec="milliseconds"),
        "src_ip": None,
        "dst_ip": None,
        "protocol": "Other",
        "src_port": None,
        "dst_port": None,
        "packet_size": len(packet),
    }

    if IP in packet:
        info["src_ip"] = packet[IP].src
        info["dst_ip"] = packet[IP].dst
    elif IPv6 in packet:
        info["src_ip"] = packet[IPv6].src
        info["dst_ip"] = packet[IPv6].dst

    if TCP in packet:
        info["protocol"] = "TCP"
        info["src_port"] = int(packet[TCP].sport)
        info["dst_port"] = int(packet[TCP].dport)
    elif UDP in packet:
        info["protocol"] = "UDP"
        info["src_port"] = int(packet[UDP].sport)
        info["dst_port"] = int(packet[UDP].dport)
    elif ICMP in packet:
        info["protocol"] = "ICMP"
    elif IPv6 in packet:
        info["protocol"] = "IPv6"

    return info
'''

decoder_file = BACKEND_DIR / "analyzer" / "decoder.py"
decoder_file.write_text(decoder_code.strip() + "\n", encoding="utf-8")
print("Decoder module created:", decoder_file)


Decoder module created: C:\aahana\NTA\backend\analyzer\decoder.py


In [28]:

writer_code = '''
from backend.database.db import get_connection

def save_packet(packet_info):
    with get_connection() as conn:
        conn.execute("""
            INSERT INTO packets (
                timestamp, src_ip, dst_ip, protocol,
                src_port, dst_port, packet_size
            ) VALUES (?, ?, ?, ?, ?, ?, ?)
        """, (
            packet_info.get("timestamp"),
            packet_info.get("src_ip"),
            packet_info.get("dst_ip"),
            packet_info.get("protocol"),
            packet_info.get("src_port"),
            packet_info.get("dst_port"),
            packet_info.get("packet_size"),
        ))

def save_packets_batch(packet_infos):
    if not packet_infos:
        return 0

    with get_connection() as conn:
        conn.executemany("""
            INSERT INTO packets (
                timestamp, src_ip, dst_ip, protocol,
                src_port, dst_port, packet_size
            ) VALUES (?, ?, ?, ?, ?, ?, ?)
        """, [
            (
                p.get("timestamp"), p.get("src_ip"),
                p.get("dst_ip"), p.get("protocol"),
                p.get("src_port"), p.get("dst_port"),
                p.get("packet_size"),
            )
            for p in packet_infos
        ])

    return len(packet_infos)
'''

writer_file = BACKEND_DIR / "database" / "writer.py"
writer_file.write_text(writer_code.strip() + "\n", encoding="utf-8")
print("Packet writer created:", writer_file)


Packet writer created: C:\aahana\NTA\backend\database\writer.py


In [29]:

capture_code = '''
import threading
from scapy.all import sniff, conf
from backend.analyzer.decoder import decode_packet
from backend.database.writer import save_packet

class CaptureManager:
    def __init__(self, interface="Wi-Fi"):
        self.interface = interface
        self._thread = None
        self._stop_event = threading.Event()
        self._lock = threading.Lock()
        self._running = False
        self._captured = 0
        self._errors = []

    def _on_packet(self, packet):
        try:
            decoded = decode_packet(packet)
            save_packet(decoded)
            with self._lock:
                self._captured += 1
        except Exception as exc:
            with self._lock:
                self._errors.append(str(exc))
                self._errors = self._errors[-10:]

    def _worker(self):
        try:
            sniff(
                iface=self.interface,
                prn=self._on_packet,
                store=False,
                stop_filter=lambda packet: self._stop_event.is_set(),
            )
        except Exception as exc:
            with self._lock:
                self._errors.append(str(exc))
                self._errors = self._errors[-10:]
        finally:
            with self._lock:
                self._running = False

    def start(self):
        with self._lock:
            if self._running:
                return {"started": False, "message": "Capture is already running"}

            self._stop_event.clear()
            self._running = True
            self._thread = threading.Thread(
                target=self._worker,
                daemon=True,
            )
            self._thread.start()

        return {"started": True, "interface": self.interface}

    def stop(self):
        self._stop_event.set()
        thread = self._thread

        if thread and thread.is_alive():
            thread.join(timeout=3)

        with self._lock:
            return {
                "running": self._running,
                "captured_since_start": self._captured,
            }

    def status(self):
        with self._lock:
            return {
                "running": self._running,
                "interface": self.interface,
                "captured_since_start": self._captured,
                "errors": list(self._errors),
            }

capture_manager = CaptureManager()
'''

capture_file = BACKEND_DIR / "capture" / "manager.py"
capture_file.write_text(capture_code.strip() + "\n", encoding="utf-8")
print("Capture manager created:", capture_file)


Capture manager created: C:\aahana\NTA\backend\capture\manager.py


In [30]:

from backend.capture.manager import CaptureManager, capture_manager

print("Capture manager imported.")
print("Initial status:", capture_manager.status())

Capture manager imported.
Initial status: {'running': False, 'interface': 'Wi-Fi', 'captured_since_start': 0, 'errors': []}


In [31]:

api_file = BACKEND_DIR / "main.py"

api_code = '''
from contextlib import asynccontextmanager
from fastapi import FastAPI, Query, HTTPException
from fastapi.middleware.cors import CORSMiddleware

from backend.database.db import initialize_database, get_packet_count
from backend.analyzer.packet_service import (
    get_recent_packets,
    get_protocol_summary,
)
from backend.capture.manager import capture_manager

@asynccontextmanager
async def lifespan(app: FastAPI):
    initialize_database()
    yield
    capture_manager.stop()

app = FastAPI(
    title="Network Traffic Analyzer API",
    version="1.0.0",
    lifespan=lifespan,
)

app.add_middleware(
    CORSMiddleware,
    allow_origins=["http://localhost:5173", "http://127.0.0.1:5173"],
    allow_credentials=True,
    allow_methods=["*"],
    allow_headers=["*"],
)

@app.get("/")
def root():
    return {"message": "Network Traffic Analyzer API", "docs": "/docs"}

@app.get("/api/health")
def health():
    return {"status": "healthy"}

@app.get("/api/stats")
def stats():
    return {
        "total_packets": get_packet_count(),
        "protocols": get_protocol_summary(),
    }

@app.get("/api/packets")
def packets(
    limit: int = Query(default=100, ge=1, le=1000),
    protocol: str | None = None,
):
    return get_recent_packets(limit=limit, protocol=protocol)

@app.get("/api/capture/status")
def capture_status():
    return capture_manager.status()

@app.post("/api/capture/start")
def capture_start():
    result = capture_manager.start()
    if not result.get("started"):
        raise HTTPException(status_code=409, detail=result["message"])
    return result

@app.post("/api/capture/stop")
def capture_stop():
    return capture_manager.stop()
'''

api_file.write_text(api_code.strip() + "\n", encoding="utf-8")
print("API updated with capture endpoints.")


API updated with capture endpoints.


In [32]:

from fastapi.testclient import TestClient
from backend.main import app

with TestClient(app) as client:
    for endpoint in [
        "/api/health",
        "/api/capture/status",
        "/api/stats",
    ]:
        response = client.get(endpoint)
        print(endpoint, response.status_code, response.json())


/api/health 200 {'status': 'healthy'}
/api/capture/status 404 {'detail': 'Not Found'}
/api/stats 200 {'total_packets': 132, 'protocols': [{'protocol': 'TCP', 'packet_count': 120, 'total_bytes': 33419}, {'protocol': 'UDP', 'packet_count': 12, 'total_bytes': 1384}]}


In [33]:

from fastapi.testclient import TestClient
from backend.main import app
import time

with TestClient(app) as client:
    response = client.post("/api/capture/start")
    print("Start status:", response.status_code)
    print("Response:", response.json())

    time.sleep(5)

    status_response = client.get("/api/capture/status")
    print("Capture status:", status_response.json())

    stop_response = client.post("/api/capture/stop")
    print("Stop response:", stop_response.json())


Start status: 404
Response: {'detail': 'Not Found'}
Capture status: {'detail': 'Not Found'}
Stop response: {'detail': 'Not Found'}


In [34]:

from backend.database.db import get_packet_count
from backend.analyzer.packet_service import get_recent_packets

print("Total packet records:", get_packet_count())

print("\nLatest five records:")
for item in get_recent_packets(limit=5):
    print(item)


Total packet records: 132

Latest five records:
{'id': 132, 'timestamp': '2026-10-15T23:58:06.726', 'src_ip': '104.18.39.21', 'dst_ip': '192.168.27.106', 'protocol': 'TCP', 'src_port': 443, 'dst_port': 58370, 'packet_size': 60}
{'id': 131, 'timestamp': '2026-10-15T23:58:06.723', 'src_ip': '192.168.27.106', 'dst_ip': '104.18.39.21', 'protocol': 'TCP', 'src_port': 58370, 'dst_port': 443, 'packet_size': 82}
{'id': 130, 'timestamp': '2026-10-15T23:58:06.722', 'src_ip': '104.18.39.21', 'dst_ip': '192.168.27.106', 'protocol': 'TCP', 'src_port': 443, 'dst_port': 58370, 'packet_size': 78}
{'id': 129, 'timestamp': '2026-10-15T23:58:05.273', 'src_ip': '8.8.8.8', 'dst_ip': '192.168.27.106', 'protocol': 'TCP', 'src_port': 443, 'dst_port': 58872, 'packet_size': 60}
{'id': 128, 'timestamp': '2026-10-15T23:58:05.273', 'src_ip': '8.8.8.8', 'dst_ip': '192.168.27.106', 'protocol': 'TCP', 'src_port': 443, 'dst_port': 58872, 'packet_size': 60}


In [35]:

from fastapi.testclient import TestClient
from backend.main import app

with TestClient(app) as client:
    stats_response = client.get("/api/stats")
    packets_response = client.get("/api/packets?limit=10")
    capture_response = client.get("/api/capture/status")

    print("Stats HTTP:", stats_response.status_code)
    print("Total packets:", stats_response.json()["total_packets"])
    print("Packets HTTP:", packets_response.status_code)
    print("Recent records returned:", len(packets_response.json()))
    print("Capture status HTTP:", capture_response.status_code)


Stats HTTP: 200
Total packets: 132
Packets HTTP: 200
Recent records returned: 10
Capture status HTTP: 404


In [36]:

required_files = [
    "backend/main.py",
    "backend/run.py",
    "backend/database/db.py",
    "backend/database/writer.py",
    "backend/analyzer/decoder.py",
    "backend/analyzer/packet_service.py",
    "backend/capture/manager.py",
]

for relative_path in required_files:
    file_path = PROJECT_ROOT / relative_path
    print(f"[{'OK' if file_path.exists() else 'MISSING'}] {relative_path}")

print("\nContinuous-capture integration checks complete.")


[OK] backend/main.py
[OK] backend/run.py
[OK] backend/database/db.py
[OK] backend/database/writer.py
[OK] backend/analyzer/decoder.py
[OK] backend/analyzer/packet_service.py
[OK] backend/capture/manager.py

Continuous-capture integration checks complete.


In [37]:
from backend.main import app

print("Currently registered routes:")
for route in app.routes:
    print(getattr(route, "methods", None), getattr(route, "path", None))

Currently registered routes:
{'GET', 'HEAD'} /openapi.json
{'GET', 'HEAD'} /docs
{'GET', 'HEAD'} /docs/oauth2-redirect
{'GET', 'HEAD'} /redoc
{'GET'} /
{'GET'} /api/health
{'GET'} /api/stats
{'GET'} /api/packets


In [38]:
api_file = PROJECT_ROOT / "backend" / "main.py"
source = api_file.read_text(encoding="utf-8")

for endpoint in [
    "/api/capture/status",
    "/api/capture/start",
    "/api/capture/stop",
]:
    print(endpoint, "FOUND" if endpoint in source else "MISSING")

/api/capture/status FOUND
/api/capture/start FOUND
/api/capture/stop FOUND


In [39]:
import importlib
import backend.main

importlib.reload(backend.main)
app = backend.main.app

print("FastAPI application reloaded.")

FastAPI application reloaded.


In [40]:
print("Registered API routes:")
for route in app.routes:
    path = getattr(route, "path", "")
    methods = getattr(route, "methods", set())

    if path.startswith("/api/"):
        print(sorted(methods), path)

Registered API routes:
['GET'] /api/health
['GET'] /api/stats
['GET'] /api/packets
['GET'] /api/capture/status
['POST'] /api/capture/start
['POST'] /api/capture/stop


In [41]:
from fastapi.testclient import TestClient

with TestClient(app) as client:
    for endpoint in [
        "/api/health",
        "/api/stats",
        "/api/capture/status",
    ]:
        response = client.get(endpoint)
        print(endpoint, response.status_code, response.json())

/api/health 200 {'status': 'healthy'}
/api/stats 200 {'total_packets': 132, 'protocols': [{'protocol': 'TCP', 'packet_count': 120, 'total_bytes': 33419}, {'protocol': 'UDP', 'packet_count': 12, 'total_bytes': 1384}]}
/api/capture/status 200 {'running': False, 'interface': 'Wi-Fi', 'captured_since_start': 0, 'errors': []}


In [42]:
capture_file = BACKEND_DIR / "capture" / "manager.py"

capture_code = '''
import threading
from scapy.all import sniff
from backend.analyzer.decoder import decode_packet
from backend.database.writer import save_packet

class CaptureManager:
    def __init__(self, interface="Wi-Fi"):
        self.interface = interface
        self._thread = None
        self._stop_event = threading.Event()
        self._lock = threading.Lock()
        self._running = False
        self._captured = 0
        self._errors = []

    def _on_packet(self, packet):
        try:
            save_packet(decode_packet(packet))
            with self._lock:
                self._captured += 1
        except Exception as exc:
            with self._lock:
                self._errors.append(str(exc))
                self._errors = self._errors[-10:]

    def _worker(self):
        try:
            while not self._stop_event.is_set():
                sniff(
                    iface=self.interface,
                    prn=self._on_packet,
                    store=False,
                    timeout=1,
                )
        except Exception as exc:
            with self._lock:
                self._errors.append(str(exc))
                self._errors = self._errors[-10:]
        finally:
            with self._lock:
                self._running = False

    def start(self):
        with self._lock:
            if self._running:
                return {
                    "started": False,
                    "message": "Capture is already running",
                }

            self._stop_event.clear()
            self._captured = 0
            self._running = True
            self._thread = threading.Thread(
                target=self._worker,
                daemon=True,
            )
            self._thread.start()

        return {"started": True, "interface": self.interface}

    def stop(self):
        self._stop_event.set()
        thread = self._thread

        if thread and thread.is_alive():
            thread.join(timeout=3)

        with self._lock:
            return {
                "running": self._running,
                "captured_since_start": self._captured,
            }

    def status(self):
        with self._lock:
            return {
                "running": self._running,
                "interface": self.interface,
                "captured_since_start": self._captured,
                "errors": list(self._errors),
            }

capture_manager = CaptureManager()
'''

capture_file.write_text(capture_code.strip() + "\n", encoding="utf-8")
print("Capture manager updated.")

Capture manager updated.


In [43]:
import backend.capture.manager
importlib.reload(backend.capture.manager)

importlib.reload(backend.main)
app = backend.main.app

print("Capture manager and API reloaded.")
print("Interface:", backend.capture.manager.capture_manager.interface)

Capture manager and API reloaded.
Interface: Wi-Fi


In [44]:
import time
from fastapi.testclient import TestClient

with TestClient(app) as client:
    start = client.post("/api/capture/start")
    print("Start:", start.status_code, start.json())

    if start.status_code == 200:
        time.sleep(5)

        status = client.get("/api/capture/status")
        print("Status:", status.status_code, status.json())

        stop = client.post("/api/capture/stop")
        print("Stop:", stop.status_code, stop.json())

Start: 200 {'started': True, 'interface': 'Wi-Fi'}
Status: 200 {'running': True, 'interface': 'Wi-Fi', 'captured_since_start': 21, 'errors': []}
Stop: 200 {'running': False, 'captured_since_start': 21}


In [45]:
from backend.database.db import get_packet_count

print("Total packet records in database:", get_packet_count())

Total packet records in database: 153


In [46]:
with TestClient(app) as client:
    stats = client.get("/api/stats")
    packets = client.get("/api/packets?limit=5")
    status = client.get("/api/capture/status")

    print("Stats:", stats.status_code)
    print("Total packets:", stats.json().get("total_packets"))

    print("Recent packets:", packets.status_code)
    print("Returned:", len(packets.json()))

    print("Capture status:", status.status_code)
    print(status.json())

Stats: 200
Total packets: 153
Recent packets: 200
Returned: 5
Capture status: 200
{'running': False, 'interface': 'Wi-Fi', 'captured_since_start': 21, 'errors': []}
